https://datos.gob.es/sites/default/files/doc/file/analisis_exploratorio_de_datos_2021.pdf
https://www.ibm.com/mx-es/think/topics/exploratory-data-analysis
https://docs.aws.amazon.com/es_es/sagemaker/latest/dg/canvas-analyses.html

## **1. DATA DESCRIPTION**

## **1.1 Prices**

1. **Temporal Scope** 

    This dataset contains historical electricity prices from 01/01/2020 to 30/09/2025.

    Data after 1 October 2025 is excluded due to the transition from hourly to 15-minute resolution, ensuring temporal consistency in the analysis.


2. **Geographical Coverage**

    The dataset covers 20 bidding zones, which can be grouped into three larger regions:
    -  <u>Baltic countries </u> 

        Lithuania, Latvia and Estonia

    -  <u>Nordic countries </u> 
        
        Denmark, Finland, Norway and Sweden

    -  <u>Central Western Europe </u> 
        
        Germany, France, the Netherlands, Belgium and Austria

    Denmark, Norway and Sweden are divided into several bidding zones.

    Other bidding zones are only present during a limited period of time and they are excluded from the analysis to ensure consistency in the geographical coverage.

    - Finland-Russia Exchange (01/01/2020 - 03/06/2020)
    - Great Britain (01/01/2020 - 31/12/2020)
    - Bulgaria (02/12/2020 - present)
    - Romania (20/11/2024 - present)

3. **Temporal Field**

    Delivery day and Delivery period (CET).


4. **Granularity**

    Hourly data.

5. **Units**

    EUR/MWh

6. **Structure**

    The dataset is stored in a wide format, where each row represents a delivery period and each bidding zone is represented as a separate column.

7. **Role in the analysis**

    This dataset represents the main target variable of the study, as it contains the electricity prices to be analyzed and modeled across time and bidding zones.


## **1.2 Volumes**

1. **Temporal Scope** 

    This dataset contains historical electricity market volumes from 01/01/2020 to 30/09/2025.

    Data after 1 October 2025 is excluded due to the transition from hourly to 15-minute resolution, ensuring temporal consistency in the analysis.


2. **Geographical Coverage**

    The dataset covers 20 bidding zones, which can be grouped into three larger regions:
    -  <u>Baltic countries </u> 

        Lithuania, Latvia and Estonia

    -  <u>Nordic countries </u> 
        
        Denmark, Finland, Norway and Sweden

    -  <u>Central Western Europe </u> 
        
        Germany, France, the Netherlands, Belgium and Austria

    Denmark, Norway and Sweden are divided into several bidding zones.

    Other bidding zones are only present during a limited period of time and they are excluded from the analysis to ensure consistency in the geographical coverage.


3. **Temporal Field**

    Delivery day and Delivery period (CET).


4. **Granularity**

    Hourly data.

5. **Units**

    MWh to ensure consistency with the price dataset, which is expressed in EUR/MWh.

6. **Structure**

    The dataset is stored in a wide format. For each bidding zone, two consecutive columns are provided, corresponding to Buy and Sell volumes.

7. **Role in the analysis**

    This dataset represents market activity in each bidding zone, reflecting supply (sell volumes) and demand (buy volumes), and may help explain electricity price dynamics.

## **1.3 Flows**

1. **Temporal Scope** 

    This dataset contains historical directional energy flows from 01/01/2020 to 30/09/2025.

    Data after 1 October 2025 is excluded due to the transition from hourly to 15-minute resolution, ensuring temporal consistency in the analysis.


2. **Geographical Coverage**

    The dataset covers directional interconnections between bidding zones included in the analysis. Each extracted table is centered on a specific delivery area and contains the corresponding import and export flows with neighboring zones.


3. **Temporal Field**

    Delivery day and Delivery period (CET).


4. **Granularity**

    Hourly data.

5. **Units**

    MWh

6. **Structure**

    The dataset is stored in a wide format, where each row represents a delivery period and each column represents a directional interconnection between two areas (e.g. NO1 -> NO2, NO2 -> NO1).

7. **Role in the analysis**

    This dataset captures cross-zonal energy exchanges and provides information about the spatial interaction between bidding zones. It may help explain price differences, market coupling, and network-related events.

## **1.4 Capacities**

1. **Temporal Scope** 

    This dataset contains historical transmission capacities from 01/01/2020 to 30/09/2025.

    Data after 1 October 2025 is excluded due to the transition from hourly to 15-minute resolution, ensuring temporal consistency in the analysis.


2. **Geographical Coverage**

    The dataset covers directional transmission capacities between bidding zones included in the analysis.

    The raw data may include additional technical nodes (e.g. sub-areas or interconnectors), which are excluded to ensure consistency with the bidding zones used in the rest of the study.


3. **Temporal Field**

    Delivery day and Delivery period (CET).


4. **Granularity**

    Hourly data.

5. **Units**

    MWh

6. **Structure**

    The dataset is stored in a wide format, where each row represents a delivery period and each column represents a directional capacity between two areas (e.g. NO1 → NO2, NO2 → NO1).

7. **Role in the analysis**

    This dataset represents the physical limits of the transmission network and provides information about the maximum possible energy exchange between zones. It is essential to identify potential congestion situations and to contextualize observed flows.

## **2. DATA LOADING**

## **2.1 Prices**

In this section, the downloaded Excel files corresponding to the Prices dataset are loaded and combined into a single raw dataframe. At this stage, no preprocessing is applied yet.

In [4]:
import os
import glob
import pandas as pd

In [5]:
# Base path for the downloaded Prices files
prices_base_path = r"C:\Users\HUGO\Desktop\Q8 - NORUEGA\TFG\tfg\NordPoool\ExcelFilesNoProcessed\Prices"

# Read all Excel files from subfolders (e.g. 2020, 2021, ...)
price_files_2020 = glob.glob(os.path.join(prices_base_path, "2020", "*.xlsx"))
price_files_2021 = glob.glob(os.path.join(prices_base_path, "2021", "*.xlsx"))

price_files = sorted(price_files_2020 + price_files_2021)

print(f"2020 files found: {len(price_files_2020)}")
print(f"2021 files found: {len(price_files_2021)}")
print(f"Total files found: {len(price_files)}")

2020 files found: 366
2021 files found: 365
Total files found: 731


Track if every price file has been downloaded and loaded correctly.

In [6]:
import os
import re
import pandas as pd

path_2021 = r"C:\Users\HUGO\Desktop\Q8 - NORUEGA\TFG\tfg\NordPoool\ExcelFilesNoProcessed\Prices\2021"

files = [f for f in os.listdir(path_2021) if f.endswith(".xlsx")]

dates_found = []
for f in files:
    match = re.search(r"\d{4}-\d{2}-\d{2}", f)
    if match:
        dates_found.append(match.group())

dates_found = pd.to_datetime(dates_found)
expected_dates = pd.date_range("2021-01-01", "2021-12-31")

missing_dates = expected_dates.difference(dates_found)

print("Missing dates:")
print(missing_dates)
print("Number of missing dates:", len(missing_dates))

Missing dates:
DatetimeIndex([], dtype='datetime64[ns]', freq='D')
Number of missing dates: 0


Loading all files.

In [7]:
def read_price_file(file):
    df = pd.read_excel(file, header=None)

    # Find the row where the actual table starts
    header_row = df[df.iloc[:, 0].astype(str).str.contains("Delivery period")].index[0]

    # Set that row as the header
    df.columns = df.iloc[header_row]

    # Remove metadata rows above the header
    df = df[(header_row + 1):]

    # Reset index after slicing
    df = df.reset_index(drop=True)

    return df


prices_dfs = []

for file in price_files:
    try:
        # Read file using custom parser
        df = read_price_file(file)

        # Add source file information for traceability
        df["source_file"] = os.path.basename(file)

        prices_dfs.append(df)

    except Exception as e:
        print(f"Error reading {file}: {e}")

# Concatenate all dataframes into a single dataset
prices_raw = pd.concat(prices_dfs, ignore_index=True)

print(f"Combined shape: {prices_raw.shape}")

Combined shape: (19737, 23)


First comprovation of the loaded data.

In [8]:
prices_raw.head()

4,Delivery period (CET),EE (EUR),LT (EUR),LV (EUR),AT (EUR),BE (EUR),FR (EUR),GER (EUR),NL (EUR),DK1 (EUR),...,NO2 (EUR),NO3 (EUR),NO4 (EUR),NO5 (EUR),SE1 (EUR),SE2 (EUR),SE3 (EUR),SE4 (EUR),source_file,PL (EUR)
0,00:00 - 01:00,13.55,13.55,13.55,22.26,22.26,22.26,22.26,22.26,4.25,...,4.25,4.25,4.25,4.25,4.25,4.25,4.25,4.25,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",NaN
1,01:00 - 02:00,7.56,7.56,7.56,20.22,20.22,20.22,20.22,20.22,4.27,...,4.27,4.27,4.27,4.27,4.27,4.27,4.27,4.27,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",NaN
2,02:00 - 03:00,7.15,7.15,7.15,19.67,19.67,19.67,19.67,19.67,4.31,...,4.31,4.31,4.31,4.31,4.31,4.31,4.31,4.31,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",NaN
3,03:00 - 04:00,8.02,8.02,8.02,19.25,19.25,19.25,19.25,19.25,4.41,...,4.41,4.41,4.41,4.41,4.41,4.41,4.41,4.41,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",NaN
4,04:00 - 05:00,7.91,7.91,7.91,19.31,19.31,19.31,19.31,19.31,4.53,...,4.53,4.53,4.53,4.53,4.53,4.53,4.53,4.53,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",NaN


In [9]:
prices_raw.tail()

4,Delivery period (CET),EE (EUR),LT (EUR),LV (EUR),AT (EUR),BE (EUR),FR (EUR),GER (EUR),NL (EUR),DK1 (EUR),...,NO2 (EUR),NO3 (EUR),NO4 (EUR),NO5 (EUR),SE1 (EUR),SE2 (EUR),SE3 (EUR),SE4 (EUR),source_file,PL (EUR)
19732,22:00 - 23:00,69.7,69.7,69.7,105.01,35.96,140.38,5.1,99,32.34,...,139.18,32.34,32.34,139.18,32.34,32.34,32.34,32.34,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",NaN
19733,23:00 - 00:00,57.98,57.98,57.98,57.98,36.34,127.81,6.32,98.2,29.76,...,138.3,29.76,29.76,138.3,29.76,29.76,29.76,29.76,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",NaN
19734,Min:,39.11,39.11,39.11,39.11,-40.16,37.33,-0.06,0,5.71,...,112.88,24.01,24.01,112.88,24.01,24.01,24.01,24.01,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",NaN
19735,Max:,105.07,105.07,105.07,105.07,92.99,193.15,62.23,142.53,105.07,...,147.24,55.36,55.36,147.24,55.36,55.36,105.07,105.07,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",NaN
19736,Average:,81.49,81.49,81.49,86.96,9.68,120.02,12.13,78.63,42.61,...,135.85,37.9,37.9,135.85,37.9,37.9,43.62,43.62,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",NaN


## **2.2 Volumes**

## **2.3 Flows**

## **2.4 Capacities**

## **3. DATA PREPROCESSING**

## **3.1 Prices**

In this section, the raw Prices dataset is cleaned and transformed in order to make it suitable for exploratory analysis. The preprocessing includes removing summary rows, cleaning column names, extracting the delivery date from the source file, and creating a proper timestamp.

In [10]:
prices = prices_raw.copy()

Next step is deleting unnecessary columns and rows.

In [11]:
# Remove summary rows that do not correspond to actual delivery periods
prices = prices[
    ~prices["Delivery period (CET)"].astype(str).str.contains("Min|Max|Average", na=False)
]

# Reset index after filtering
prices = prices.reset_index(drop=True)

And the following step is clean column names.

In [12]:
# Clean column names by removing the currency suffix
prices.columns = [
    col.replace(" (EUR)", "") if isinstance(col, str) else col
    for col in prices.columns
]

Extract delivery date from the source file name and create a proper timestamp.

In [13]:
# Extract delivery day from the source file name
prices["Delivery day"] = prices["source_file"].str.extract(r"(\d{4}-\d{2}-\d{2})")

# Convert delivery day to datetime
prices["Delivery day"] = pd.to_datetime(prices["Delivery day"])

# Extract the starting hour from the delivery period
prices["hour"] = prices["Delivery period (CET)"].str[:2].astype(int)

# Create a full timestamp combining delivery day and hour
prices["timestamp"] = prices["Delivery day"] + pd.to_timedelta(prices["hour"], unit="h")

# Sort dataset chronologically
prices = prices.sort_values("timestamp").reset_index(drop=True)

In [14]:
prices.head()

,Delivery period (CET),EE,LT,LV,AT,BE,FR,GER,NL,DK1,...,NO5,SE1,SE2,SE3,SE4,source_file,PL,Delivery day,hour,timestamp
0,00:00 - 01:00,28.78,28.78,28.78,41.88,41.88,41.88,41.88,41.88,33.42,...,31.82,28.78,28.78,28.78,28.78,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",NaN,2020-01-01,0,2020-01-01 00:00:00
1,01:00 - 02:00,28.45,28.45,28.45,38.6,38.6,38.6,38.6,38.6,31.77,...,31.77,28.45,28.45,28.45,28.45,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",NaN,2020-01-01,1,2020-01-01 01:00:00
2,02:00 - 03:00,27.9,27.9,27.9,36.55,36.55,36.55,36.55,36.55,31.57,...,31.57,27.9,27.9,27.9,27.9,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",NaN,2020-01-01,2,2020-01-01 02:00:00
3,03:00 - 04:00,27.52,27.52,27.52,32.32,32.32,32.32,32.32,32.32,31.28,...,31.28,27.52,27.52,27.52,27.52,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",NaN,2020-01-01,3,2020-01-01 03:00:00
4,04:00 - 05:00,27.54,27.54,27.54,30.85,30.85,30.85,30.85,30.85,30.85,...,30.72,27.54,27.54,27.54,27.54,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",NaN,2020-01-01,4,2020-01-01 04:00:00


In [15]:
prices.columns.tolist()

['Delivery period (CET)',
 'EE',
 'LT',
 'LV',
 'AT',
 'BE',
 'FR',
 'GER',
 'NL',
 'DK1',
 'DK2',
 'FI',
 'NO1',
 'NO2',
 'NO3',
 'NO4',
 'NO5',
 'SE1',
 'SE2',
 'SE3',
 'SE4',
 'source_file',
 'PL',
 'Delivery day',
 'hour',
 'timestamp']

Even we decided to exclude some zones to ensure consistency in the geographical coverage, there are few data from PL we need to remove to avoid inconsistencies in the analysis.

In [16]:
# Drop unwanted zones
prices = prices.drop(columns=["PL"], errors="ignore")

In [17]:
prices.head()

,Delivery period (CET),EE,LT,LV,AT,BE,FR,GER,NL,DK1,...,NO4,NO5,SE1,SE2,SE3,SE4,source_file,Delivery day,hour,timestamp
0,00:00 - 01:00,28.78,28.78,28.78,41.88,41.88,41.88,41.88,41.88,33.42,...,28.78,31.82,28.78,28.78,28.78,28.78,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",2020-01-01,0,2020-01-01 00:00:00
1,01:00 - 02:00,28.45,28.45,28.45,38.6,38.6,38.6,38.6,38.6,31.77,...,28.45,31.77,28.45,28.45,28.45,28.45,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",2020-01-01,1,2020-01-01 01:00:00
2,02:00 - 03:00,27.9,27.9,27.9,36.55,36.55,36.55,36.55,36.55,31.57,...,27.9,31.57,27.9,27.9,27.9,27.9,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",2020-01-01,2,2020-01-01 02:00:00
3,03:00 - 04:00,27.52,27.52,27.52,32.32,32.32,32.32,32.32,32.32,31.28,...,27.52,31.28,27.52,27.52,27.52,27.52,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",2020-01-01,3,2020-01-01 03:00:00
4,04:00 - 05:00,27.54,27.54,27.54,30.85,30.85,30.85,30.85,30.85,30.85,...,27.54,30.72,27.54,27.54,27.54,27.54,"auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...",2020-01-01,4,2020-01-01 04:00:00


Last verification of the preprocessed data.

In [18]:
prices.shape
prices[["timestamp", "Delivery day", "Delivery period (CET)"]].head()
prices.isna().sum().sort_values(ascending=False).head(10)

Delivery period (CET)    0
NO2                      0
hour                     0
Delivery day             0
source_file              0
SE4                      0
SE3                      0
SE2                      0
SE1                      0
NO5                      0
dtype: int64

## **3.2 Volumes**

## **3.3 Flows**

## **3.4 Capacities**

## **4. EXPLORATORY DATA ANALYSIS (EDA)**

## **4.1 Prices**

**4.1 Initial validation of the dataset**

In [46]:
prices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17544 entries, 0 to 17543
Data columns (total 25 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Delivery period (CET)  17544 non-null  object        
 1   EE                     17544 non-null  float64       
 2   LT                     17544 non-null  float64       
 3   LV                     17544 non-null  float64       
 4   AT                     17544 non-null  float64       
 5   BE                     17544 non-null  float64       
 6   FR                     17544 non-null  float64       
 7   GER                    17544 non-null  float64       
 8   NL                     17544 non-null  float64       
 9   DK1                    17544 non-null  float64       
 10  DK2                    17544 non-null  float64       
 11  FI                     17544 non-null  float64       
 12  NO1                    17544 non-null  float64       
 13  N

In [47]:
print(prices.shape)

(17544, 25)


In [48]:
print(prices.head())

  Delivery period (CET)     EE     LT     LV     AT     BE     FR    GER  \
0         00:00 - 01:00  28.78  28.78  28.78  41.88  41.88  41.88  41.88   
1         01:00 - 02:00  28.45  28.45  28.45  38.60  38.60  38.60  38.60   
2         02:00 - 03:00  27.90  27.90  27.90  36.55  36.55  36.55  36.55   
3         03:00 - 04:00  27.52  27.52  27.52  32.32  32.32  32.32  32.32   
4         04:00 - 05:00  27.54  27.54  27.54  30.85  30.85  30.85  30.85   

      NL    DK1  ...    NO4    NO5    SE1    SE2    SE3    SE4  \
0  41.88  33.42  ...  28.78  31.82  28.78  28.78  28.78  28.78   
1  38.60  31.77  ...  28.45  31.77  28.45  28.45  28.45  28.45   
2  36.55  31.57  ...  27.90  31.57  27.90  27.90  27.90  27.90   
3  32.32  31.28  ...  27.52  31.28  27.52  27.52  27.52  27.52   
4  30.85  30.85  ...  27.54  30.72  27.54  27.54  27.54  27.54   

                                         source_file  Delivery day  hour  \
0  auction_prices_Day-ahead_EE,LT,LV,AT,BE,FR,GER...    2020-01-01    

In [49]:
print(prices.isnull().sum())

Delivery period (CET)    0
EE                       0
LT                       0
LV                       0
AT                       0
BE                       0
FR                       0
GER                      0
NL                       0
DK1                      0
DK2                      0
FI                       0
NO1                      0
NO2                      0
NO3                      0
NO4                      0
NO5                      0
SE1                      0
SE2                      0
SE3                      0
SE4                      0
source_file              0
Delivery day             0
hour                     0
timestamp                0
dtype: int64


The processed prices dataset contains 17,544 rows and 25 columns, indicating that the data was successfully loaded and consolidated after preprocessing. The initial inspection shows that all variables contain the same number of non-null observations, which confirms the absence of missing values in the final dataset.

The temporal structure is also correctly defined, including variables such as Delivery day, hour, and timestamp, which will facilitate the subsequent time-based analysis of electricity prices.

In addition, the columns corresponding to market prices contain valid numerical values, although they are currently stored as object rather than numeric data types. Therefore, these variables may require conversion before performing statistical calculations in the following stages of the exploratory analysis.

Conversion of price to numeric data (float):

In [50]:
price_cols = ['EE', 'LT', 'LV', 'AT', 'BE', 'FR', 'GER', 'NL', 'DK1', 'DK2',
              'FI', 'NO1', 'NO2', 'NO3', 'NO4', 'NO5', 'SE1', 'SE2', 'SE3', 'SE4']

prices[price_cols] = prices[price_cols].astype(float)

**4.2 Descriptive analysis**

<u>4.2.1 Descriptive statistics by market area</u>

First of all we will analyze based on each bidding zone.

In [51]:
descriptive_stats = prices[price_cols].describe().T
print(descriptive_stats)

       count       mean        std     min      25%     50%      75%      max
EE   17544.0  60.171799  57.015366   -1.73  26.9300  46.700  75.1625  1000.07
LT   17544.0  62.207674  58.984931   -1.73  27.9800  47.590  77.1025  1000.07
LV   17544.0  61.375862  57.749738   -1.73  27.3375  46.870  76.9025  1000.07
AT   17544.0  69.947278  67.021554  -77.68  31.9200  47.645  79.7625   620.00
BE   17544.0  67.951558  67.772925 -115.31  29.3075  45.975  78.4275   620.00
FR   17544.0  70.634598  71.836834  -75.82  30.0975  46.900  79.1575   620.00
GER  17544.0  63.614913  62.972404  -83.94  28.8800  45.160  76.3150   620.00
NL   17544.0  67.553277  64.457033  -79.19  30.0975  45.600  79.1950   620.00
DK1  17544.0  56.517318  56.961436  -58.80  21.5000  41.455  74.4400   620.00
DK2  17544.0  58.119936  58.073287  -42.66  22.4275  42.900  73.6900   626.06
FI   17544.0  50.151072  53.745514   -1.73  19.7000  38.490  62.7800  1000.07
NO1  17544.0  41.942567  47.357799   -1.97   6.8600  27.360  58.

Basic descriptive statistics were computed for each market area included in the dataset in order to obtain an initial overview of electricity price behaviour. In all cases, each zone contains 17,544 observations, which indicates a homogeneous temporal coverage across all the analysed areas.

The results reveal significant differences in average price levels between market zones. Central European markets such as France, Austria, Belgium, the Netherlands, and Germany show the highest mean values, whereas several Nordic areas, particularly NO3, NO4, SE1, and SE2, display considerably lower average prices.

Likewise, price dispersion is not uniform across markets. Some zones, such as Belgium, France, Austria, and Germany, present higher standard deviations, reflecting greater variability in price behaviour throughout the analysed period. By contrast, other areas, mainly certain Nordic zones, show a more stable pattern.

Finally, the statistics reveal the presence of negative minimum prices in several market areas, as well as particularly high maximum values in some markets. This suggests the existence of extreme market episodes, which will be examined in greater detail in the following sections.

<u>4.2.2 Analysis by country</u>

In the following step, we will analyze the price behaviour in countries with multiple bidding zones:

In [52]:
prices["NO"] = prices[["NO1", "NO2", "NO3", "NO4", "NO5"]].mean(axis=1)
prices["SE"] = prices[["SE1", "SE2", "SE3", "SE4"]].mean(axis=1)
prices["DK"] = prices[["DK1", "DK2"]].mean(axis=1)

In [55]:
country_cols = ["DK", "NO", "SE"]

In [56]:
country_stats = prices[country_cols].describe().T
print(country_stats)

      count       mean        std     min        25%     50%        75%  \
DK  17544.0  57.318627  56.995218 -48.935  22.450000  42.195  73.035000   
NO  17544.0  34.620706  35.596080   0.000   6.789000  25.850  49.881000   
SE  17544.0  38.396537  35.455782  -1.970  14.988125  29.205  51.634375   

        max  
DK  620.000  
NO  383.052  
SE  388.050  


Comparison between bidding zones and average price by country.

- Norway

In [58]:
norway_comparison = []

for zone in ["NO1", "NO2", "NO3", "NO4", "NO5"]:
    zone_mean = prices[zone].mean()
    country_mean = prices["NO"].mean()
    diff_mean = (prices[zone] - prices["NO"]).mean()
    
    norway_comparison.append({
        "Zone": zone,
        "Zone mean": round(zone_mean, 2),
        "Norway mean": round(country_mean, 2),
        "Mean difference vs Norway": round(diff_mean, 2)
    })

norway_comparison_df = pd.DataFrame(norway_comparison)
print(norway_comparison_df)

  Zone  Zone mean  Norway mean  Mean difference vs Norway
0  NO1      41.94        34.62                       7.32
1  NO2      42.15        34.62                       7.53
2  NO3      25.24        34.62                      -9.38
3  NO4      21.94        34.62                     -12.68
4  NO5      41.84        34.62                       7.21


As can be observed, three bidding zones (NO1, NO2 and NO5) present a similar average price, approximately 20% above the national mean. In contrast, NO3 and NO4 show significantly lower average prices, standing around 30% and 35% below the country’s average, respectively.

This pattern suggests that the Norwegian electricity market is not homogeneous across bidding zones. A possible explanation is that the zones with higher average prices are located in the southern part of the country, where electricity demand is generally higher due to greater population concentration and economic activity. By contrast, NO3 and NO4 are situated in the northern part of Norway, where lower demand and a stronger presence of renewable generation, particularly hydropower and wind power, may contribute to lower price levels.

- Sweden

In [59]:
sweden_comparison = []

for zone in ["SE1", "SE2", "SE3", "SE4"]:
    zone_mean = prices[zone].mean()
    country_mean = prices["SE"].mean()
    diff_mean = (prices[zone] - prices["SE"]).mean()
    
    sweden_comparison.append({
        "Zone": zone,
        "Zone mean": round(zone_mean, 2),
        "Sweden mean": round(country_mean, 2),
        "Mean difference vs Sweden": round(diff_mean, 2)
    })

sweden_comparison_df = pd.DataFrame(sweden_comparison)
print(sweden_comparison_df)

  Zone  Zone mean  Sweden mean  Mean difference vs Sweden
0  SE1      28.42         38.4                      -9.98
1  SE2      28.45         38.4                      -9.95
2  SE3      43.56         38.4                       5.17
3  SE4      53.15         38.4                      14.76


A pattern similar to that observed in Norway can also be identified in Sweden. The northern bidding zones, SE1 and SE2, show average prices clearly below the national mean, whereas SE3 and especially SE4 remain above the country’s average. In particular, SE4 stands out as the most expensive Swedish market area, while SE1 and SE2 record the lowest average prices.

This result suggests that the Swedish electricity market is also characterised by significant internal heterogeneity across bidding zones. A possible explanation is that the northern zones, which are less densely populated and have strong access to electricity generation resources, tend to exhibit lower prices, while the southern zones are associated with higher demand and therefore higher average price levels.

- Denmark

In [60]:
denmark_comparison = []

for zone in ["DK1", "DK2"]:
    zone_mean = prices[zone].mean()
    country_mean = prices["DK"].mean()
    diff_mean = (prices[zone] - prices["DK"]).mean()
    
    denmark_comparison.append({
        "Zone": zone,
        "Zone mean": round(zone_mean, 2),
        "Denmark mean": round(country_mean, 2),
        "Mean difference vs Denmark": round(diff_mean, 2)
    })

denmark_comparison_df = pd.DataFrame(denmark_comparison)
print(denmark_comparison_df)

  Zone  Zone mean  Denmark mean  Mean difference vs Denmark
0  DK1      56.52         57.32                        -0.8
1  DK2      58.12         57.32                         0.8


In contrast to Norway and Sweden, Denmark shows a much more homogeneous pattern across its bidding zones. DK1 remains only slightly below the national average, while DK2 is only marginally above it, with mean differences of approximately -0.8 and +0.8 respectively.

These results indicate that price differences between Danish bidding zones are relatively limited, suggesting a more integrated internal market structure. Therefore, in the case of Denmark, the aggregated country price appears to be a good representation of the overall behaviour of the national market.

**4.3 Temporal Behaviour of Electricity Prices**